# Task06 — Colab GPU smoke test (phase2-v1)

Chọn **Runtime → Change runtime type → GPU**, rồi chạy các cell theo thứ tự.
Notebook lấy repo và gọi CLI Task05: **ViSoBERT / DFT / SA / NEU-ESC train 10% / seed 0 / LR 3e-5**, dev gốc đầy đủ, BF16 strict, tối đa 10 epoch, patience 2. Không chấm test hay dò LR.

Mặc định lấy bản mới nhất của nhánh `phase2-resource-transfer` trên GitHub, sau đó cố định checkout tại commit vừa lấy trong suốt lượt chạy. Repo riêng tư: bật `PRIVATE_REPO` và thêm secret Colab `GITHUB_TOKEN` có quyền đọc repo. Không dán token vào URL.
Để resume đúng phiên bản sau khi nhánh đã có commit mới, đặt `REPO_REF` bằng commit được ghi trong `environment.json` của lượt cũ. Output được tách theo commit.

Mặc định thử ngắt sau resume đầu tiên rồi tiếp tục, cuối cùng kiểm tra SKIP. Smoke không xuất checkpoint main encoder/head; `work/<run_id>/resume.pt` giữ model hiện tại và best weights để phục hồi. Không dùng nó thay checkpoint DFT main cho C0.

Output mặc định trên ổ tạm `/content`: tải ZIP trước khi đóng runtime. Resume lưu ở update đầu tiên và mỗi lần eval (mặc định cuối epoch). Nếu bị ngắt giữa hai lần lưu, các update chưa lưu sẽ được chạy lại. Có thể bật `USE_DRIVE` để lưu output/resume vào Drive; ghi file lớn trên Drive có thể chậm hơn. Notebook này chưa được chạy GPU trong quá trình tạo.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, signal, shutil

REPO_URL = "https://github.com/PhuongNgoVan/Xdomain-Transfer-Learning.git"
REPO_REF = "phase2-resource-transfer"  # branch name; use a commit SHA to reproduce an older run
REPO_DIR = Path("/content/xdomain-task06")
PRIVATE_REPO = False
USE_DRIVE = False
VERIFY_RESUME = True
DOWNLOAD_ZIP = True
INCLUDE_RESUME_IN_ZIP = False  # b?t n?u c?n mang resume.pt sang runtime kh?c (file l?n)

assert Path("/content").is_dir(), "This notebook requires Google Colab."
OUTPUT_BASE = Path("/content/task06_outputs")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/Xdomain/Task06")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)


## 1. Kiểm tra GPU
Dừng nếu CUDA hoặc BF16 không có. Không tự hạ precision hoặc đổi giao thức.


In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a CUDA GPU runtime in Colab."
assert torch.cuda.is_bf16_supported(), "BF16 unsupported: select another GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Preinstalled Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| BF16: True")
subprocess.run(["nvidia-smi"], check=True)


## 2. Lấy repo từ nhánh đã chọn
Fetch nhánh `REPO_REF`, kiểm tra checkout sạch và cố định tại commit vừa fetch để code không đổi giữa train/resume. Không reset hay ghi đè file đang sửa. Output nằm ngoài checkout.


In [ ]:
import base64

def git_command(args, cwd=None):
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    if PRIVATE_REPO:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        assert token, "Missing GITHUB_TOKEN secret."
        auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0="Authorization: Basic " + auth)
    result = subprocess.run(["git", *args], cwd=cwd, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode:
        raise RuntimeError("Git failed: check URL, read permission, and whether the commit was pushed.\n" + result.stderr)
    return result.stdout.strip()

if not REPO_DIR.exists():
    git_command(["clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)])
if not (REPO_DIR / ".git").is_dir():
    raise RuntimeError("REPO_DIR exists but is not a Git checkout; choose another path.")
# Recover an empty clone created by the earlier --no-checkout notebook.
# Only populate it when no worktree files or index entries exist.
if not git_command(["ls-files"], REPO_DIR) and all(p.name == ".git" for p in REPO_DIR.iterdir()):
    git_command(["restore", "--source=HEAD", "--staged", "--worktree", "--", "."], REPO_DIR)
if git_command(["diff", "--name-only"], REPO_DIR) or git_command(["diff", "--cached", "--name-only"], REPO_DIR):
    raise RuntimeError("Checkout has changes; choose a fresh REPO_DIR.")
git_command(["fetch", "--depth=1", "origin", REPO_REF], REPO_DIR)
git_command(["checkout", "--detach", "FETCH_HEAD"], REPO_DIR)
COMMIT = git_command(["rev-parse", "HEAD"], REPO_DIR)
print("Commit:", COMMIT)
OUTPUT_DIR = OUTPUT_BASE / COMMIT[:12]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = OUTPUT_DIR / "results" / "phase2"
for name in ["configs/phase2.yaml", "configs/phase2_smoke_example.yaml", "src/phase2/cli.py",
             "data/clean/neuesc_100.csv", "data/subsets_phase2/MANIFEST.json",
             "data/subsets_phase2/neuesc_f10_s0_ids.csv"]:
    assert (REPO_DIR / name).is_file(), f"Missing from commit: {name}"
print("Output:", OUTPUT_DIR)


## 3. Môi trường Python riêng
Giữ PyTorch CUDA của Colab, tạo venv `--system-site-packages`, cài các dependency cần cho pipeline theo pin trong `requirements.txt`. Không cài Java/VnCoreNLP vì dùng văn bản thô. Nếu pin không cài được, cell báo lỗi, không tự thay phiên bản.

Ghi Torch thực tế và danh sách package vào output; khác biệt với pin Torch của repo cần được ghi nhận khi so sánh môi trường. Smoke chưa chứng nhận tái lập baseline lịch sử.


In [ ]:
VENV_DIR = Path("/content/task06-venv")
PYTHON = VENV_DIR / "bin" / "python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV_DIR)], check=True)
wanted = {"transformers", "huggingface-hub", "scikit-learn", "pandas", "numpy",
          "PyYAML", "sentencepiece", "protobuf"}
pins = [line.strip() for line in (REPO_DIR / "requirements.txt").read_text().splitlines()
        if line.strip().split("==")[0] in wanted]
assert len(pins) == len(wanted), "Missing required dependency pins in repository."
install = subprocess.run([str(PYTHON), "-m", "pip", "install", *pins], text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(OUTPUT_DIR / "install.log").write_text(install.stdout, encoding="utf-8")
print(install.stdout[-5000:])
install.check_returncode()
RUN_ENV = os.environ.copy()
RUN_ENV.update(PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_TELEMETRY="1")
RUN_ENV.pop("HF_HUB_OFFLINE", None)
RUN_ENV.pop("TRANSFORMERS_OFFLINE", None)

def run_python(source):
    result = subprocess.run([str(PYTHON), "-u", "-c", source], cwd=REPO_DIR, env=RUN_ENV,
                            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()
    return result.stdout

run_python("from src.phase2.run import run_phase2_experiment; import torch; "
           "assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(); "
           "print('Pipeline imports and CUDA/BF16: PASS')")
freeze = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
(OUTPUT_DIR / "pip-freeze.txt").write_text(freeze, encoding="utf-8")


## 4. Kiểm tra dữ liệu và giao thức
Chỉ chuẩn bị train/dev SA. NEU-ESC lưu split chung một CSV: lọc train/dev trước tiền xử lý, không truyền test vào Trainer. Không tạo lại hoặc sửa subset/manifest.


In [ ]:
preflight = r"""
import json, platform, subprocess
from pathlib import Path
import torch
from src.phase2.protocol import load_phase2_config
from src.phase2.data import load_phase2_prepared_data
from src.common.hashing import sha256_file
root = Path.cwd()
cfg = load_phase2_config()
data = load_phase2_prepared_data('SA', 10, 0, 'ViSoBERT', cfg, run_mode='smoke')
assert len(data['train_df']) == 2305 and len(data['dev_df']) == 3305
assert data['test_df'] is data['test_texts'] is data['test_labels'] is None
assert cfg['training']['strict_precision'] and cfg['training']['precision'] == 'bf16'
frozen = [p for folder in ('data/clean', 'data/splits', 'data/subsets_phase2', 'src/seq', 'src/zs', 'src/stats')
          for p in (root / folder).rglob('*') if p.is_file() and '__pycache__' not in p.parts]
frozen += [root / p for p in ('configs/base.yaml', 'configs/zs.yaml', 'configs/seq.yaml',
                             'configs/phase2.yaml', 'src/common/constants.py')]
output = Path(__OUTPUT__)
(output / 'frozen_before.json').write_text(json.dumps({str(p.relative_to(root)): sha256_file(p) for p in frozen}))
info = {'commit': __COMMIT__, 'python': platform.python_version(), 'torch': torch.__version__,
        'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
        'bf16_supported': torch.cuda.is_bf16_supported(), 'train_rows': len(data['train_df']),
        'dev_rows': len(data['dev_df']), 'data_provenance': data['provenance'],
        'nvidia_smi': subprocess.check_output(['nvidia-smi'], text=True)}
(output / 'environment.json').write_text(json.dumps(info, ensure_ascii=False, indent=2))
print('Preflight PASS: train=2305, dev=3305, no test prepared.')
"""
run_python(preflight.replace('__OUTPUT__', repr(str(OUTPUT_DIR))).replace('__COMMIT__', repr(COMMIT)))


## 5. Chạy smoke và kiểm tra resume
Cell này tải ViSoBERT và train GPU khi thực thi trên Colab. Giữ cấu hình Task06, không dùng cắt mẫu, đổi epoch hay precision fallback.

Nếu `VERIFY_RESUME=True` và chưa có resume, chờ `resume.pt` đầu tiên, ngắt tiến trình rồi gọi lại cùng lệnh. Nếu đã có resume, CLI tiếp tục từ đó. Không sửa code/config giữa hai lần gọi. Khi lỗi, xem `train.log` trước khi nghiệm thu.


In [ ]:
COMMAND = [str(PYTHON), "-u", "-m", "src.phase2.cli", "--config", "configs/phase2_smoke_example.yaml",
           "--results-dir", str(RESULTS_DIR), "--ckpt-out-dir", str(OUTPUT_DIR / "checkpoints" / "phase2")]
LOG_PATH = OUTPUT_DIR / "train.log"
RESUME_CHECK_PATH = OUTPUT_DIR / "resume_check.json"

def stop_process(proc):
    if proc.poll() is None:
        os.killpg(proc.pid, signal.SIGINT)
        try:
            proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait(timeout=30)

def launch(interrupt_at_resume=False):
    with LOG_PATH.open("a", encoding="utf-8") as log:
        log.write("\n--- CLI start " + time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()) + " ---\n")
        log.flush()
        proc = subprocess.Popen(COMMAND, cwd=REPO_DIR, env=RUN_ENV, stdout=log,
                                stderr=subprocess.STDOUT, start_new_session=True)
        last_print = 0
        try:
            while proc.poll() is None:
                time.sleep(3)
                resumes = list((RESULTS_DIR / "work").glob("*/resume.pt"))
                if interrupt_at_resume and resumes:
                    stop_process(proc)
                    if proc.returncode == 0:
                        print("CLI finished before interruption; resume trial not recorded.")
                        return False
                    RESUME_CHECK_PATH.write_text(json.dumps({'interrupted': True, 'resume_file': str(resumes[0])}))
                    print("Interrupted after resume checkpoint; continuing with the same CLI.")
                    return True
                if time.monotonic() - last_print > 30:
                    last_print = time.monotonic()
                    print("Running...", "| resume exists:", bool(resumes))
                    if LOG_PATH.exists():
                        print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-800:])
            if proc.returncode:
                print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-6000:])
                raise RuntimeError(f"CLI failed (exit={proc.returncode}); xem {LOG_PATH}")
        except BaseException:
            stop_process(proc)
            raise
    return False

existing_resume = list((RESULTS_DIR / "work").glob("*/resume.pt"))
if VERIFY_RESUME and not existing_resume:
    was_interrupted = launch(interrupt_at_resume=True)
    if was_interrupted:
        launch()
        check = json.loads(RESUME_CHECK_PATH.read_text())
        check['resumed_to_completion'] = True
        RESUME_CHECK_PATH.write_text(json.dumps(check, indent=2))
else:
    launch()
print(LOG_PATH.read_text(encoding="utf-8", errors="replace")[-3000:])


## 6. Kiểm tra SKIP và tạo báo cáo
Gọi lại CLI phải trả SKIP. Kiểm tra SHA-256 artifact, xác suất/nhãn dev, loss, GPU/BF16, trạng thái resume và tài nguyên đóng băng. Chỉ tạo báo cáo sau khi các kiểm tra đạt.


In [ ]:
skip = subprocess.run(COMMAND, cwd=REPO_DIR, env=RUN_ENV, text=True,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(OUTPUT_DIR / "skip_check.log").write_text(skip.stdout, encoding="utf-8")
print(skip.stdout)
skip.check_returncode()
assert ": SKIP" in skip.stdout, "Repeated CLI did not SKIP; inspect fingerprint and artifacts."


In [ ]:
verify = r"""
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from src.common.hashing import sha256_file
from src.common.schema import ZS_PRED_COLUMNS
from src.phase2.protocol import compute_dev_selection_score
output = Path(__OUTPUT__)
results = output / 'results' / 'phase2'
manifests = list((results / 'runs').glob('*/completion.json'))
assert len(manifests) == 1, 'Output must be dedicated to one smoke run.'
manifest = manifests[0]
root = manifest.parent
completion = json.loads(manifest.read_text())
assert completion['complete'] is True and completion['ckpt_path'] is None
for rel, digest in completion['artifacts'].items():
    assert sha256_file((root / rel).resolve()) == digest, rel
assert not list(results.rglob('predictions_test.csv'))
meta = json.loads((root / 'metadata.json').read_text())
assert (meta['run_mode'], meta['method'], meta['direction'], meta['fraction'], meta['seed']) == ('smoke', 'dft', 'SA', 10, 0)
assert meta['git_commit'] == __COMMIT__ and meta['max_examples'] is None
metrics = json.loads((root / 'metrics.json').read_text())
cost = json.loads((root / 'cost.json').read_text())
assert metrics['test_metrics'] is None and metrics['history']
assert all(math.isfinite(h['train_loss']) for h in metrics['history'])
assert cost['effective_precision'] == 'bf16' and cost['n_gpus'] == 1
assert cost['train_seconds'] > 0 and cost['peak_vram_mb'] > 0
assert cost['total_params'] == cost['trainable_params'] > 0
pred = pd.read_csv(root / 'predictions_dev.csv')
assert tuple(pred.columns) == ZS_PRED_COLUMNS and len(pred) == 3305 and pred.id.is_unique
p = pred[['p3_neg','p3_neu','p3_pos']].to_numpy()
assert np.isfinite(p).all() and (p >= 0).all() and np.allclose(p.sum(1), 1, atol=1e-6)
assert np.isclose(compute_dev_selection_score(pred.y_true3, p.argmax(1), 'neuesc'), metrics['dev_selection_score'])
rows = pd.read_csv(results / 'phase2_runs.csv')
assert len(rows) == 1 and rows.split.eq('dev').all() and rows.label_view.eq('V3-NEU').all()
state = torch.load(results / 'work' / meta['run_id'] / 'resume.pt', map_location='cpu', weights_only=False)
for key in ('model','optimizer','scheduler','epoch','cursor','step','patience','best_score','best_step',
            'best_epoch','best_state','history','torch_rng','numpy_rng','python_rng','cuda_rng'):
    assert key in state, key
assert state['best_state'] is not None
assert state['best_step'] == metrics['best_step'] and state['step'] == cost['actual_update_steps']
assert state['history'] == metrics['history']
del state
frozen = json.loads((output / 'frozen_before.json').read_text())
assert all(sha256_file(Path(name)) == digest for name, digest in frozen.items())
resume_check = json.loads((output / 'resume_check.json').read_text()) if (output / 'resume_check.json').exists() else {}
resume_pass = bool(resume_check.get('interrupted') and resume_check.get('resumed_to_completion'))
report = '\n'.join([
    '# Task06 — GPU smoke test', '',
    '- Commit: ' + meta['git_commit'], '- Run ID: ' + meta['run_id'],
    '- Configuration: ViSoBERT / DFT / SA / train 10% / seed0 / LR3e-5 / BF16 strict.',
    '- Data: train 2305, dev 3305; test not scored.',
    '- GPU: ' + cost['gpu_name'],
    '- Dev macro-F1 V3: ' + str(metrics['dev_selection_score']),
    '- Best epoch / step: ' + str(metrics['best_epoch']) + ' / ' + str(metrics['best_step']),
    '- Actual update steps: ' + str(cost['actual_update_steps']),
    '- Train seconds: ' + str(cost['train_seconds']),
    '- GPU hours: ' + str(cost['gpu_hours']),
    '- Peak VRAM (MiB): ' + str(cost['peak_vram_mb']),
    '- Total / trainable params: ' + str(cost['total_params']) + ' / ' + str(cost['trainable_params']),
    '- Truncation rate: ' + str(cost['trunc_rate']),
    '- Artifact hashes, dev probabilities, finite loss, frozen files, SKIP: PASS.',
    '- Resume fields/best/history: PASS; GPU interruption and continuation: ' + ('PASS' if resume_pass else 'NOT VERIFIED in this session'),
    '', 'Smoke has no main encoder/head export; resume.pt contains best weights.',
    'Trainer cost: peak VRAM covers the final fit invocation, not all interrupted processes.',
    'train_seconds excludes model downloads, setup, ZIP and unsaved work after the interrupted checkpoint.',
    'DFT GPU hours do not precisely estimate Seq/C0/LoRA cost.',
    'Actual environment is recorded in environment.json and pip-freeze.txt; historical baselines remain unverified.',
    'Review logs and measurements before Task07; no LR selection from test.',
]) + '\n'
report_path = output / 'reports' / 'phase2' / 'smoke_test.md'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(report, encoding='utf-8')
print(report)
"""
run_python(verify.replace('__OUTPUT__', repr(str(OUTPUT_DIR))).replace('__COMMIT__', repr(COMMIT)))


## 7. Xuất kết quả
ZIP gồm report, log, metadata/config, metrics/cost, prediction dev, provenance và môi trường. `resume.pt` lớn: bật `INCLUDE_RESUME_IN_ZIP` nếu cần chuyển runtime. Để phục hồi, giữ đúng commit, môi trường, dữ liệu và đường dẫn output.

Nếu chạy lỗi vẫn có thể chạy cell này để tải log chẩn đoán. ZIP chứa log lỗi không có nghĩa Task06 hoàn thành.


In [ ]:
import zipfile
archive = Path("/content") / f"task06_{COMMIT[:12]}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if not path.is_file() or path.suffix == ".tmp":
            continue
        if path.name == "resume.pt" and not INCLUDE_RESUME_IN_ZIP:
            continue
        bundle.write(path, path.relative_to(OUTPUT_DIR))
print("ZIP:", archive, "| MiB:", round(archive.stat().st_size / 1024**2, 2))
print("Report:", OUTPUT_DIR / "reports" / "phase2" / "smoke_test.md")
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(archive))
